# Streamlit — Build Data Apps with Plain Python

> **Goal:** learn Streamlit from the first title to dashboards, uploads, caching, session state, machine learning, multipage apps, and deployment.

## Explain it like I am 5

Imagine Python has a box of webpage building blocks. `st.title()` places a big title block. `st.button()` places a button block. `st.dataframe()` places a table block. You stack blocks from top to bottom, and Streamlit builds the page.

This notebook expands the original introduction and points to runnable `.py` examples in this folder. A notebook is ideal for reading and calculations; launch real Streamlit interfaces with `streamlit run file.py`.

## Learning map

| Level | What you will learn |
|---|---|
| Basic | run an app, text, widgets, buttons, inputs |
| Intermediate | forms, layouts, files, tables, charts, state, caching |
| Practical | dashboards, ML, multipage structure, themes, deployment |

By the end, you should understand not only *what* to type, but why a Streamlit app reruns and where each kind of data should live.

## 1. Setup

From the `14-Streamlit` folder:

```text
python -m venv .venv
# Windows PowerShell
.venv\Scripts\Activate.ps1
python -m pip install -r requirements.txt
streamlit run app.py
```

The terminal prints a local URL. Open it in a browser and stop the server with `Ctrl+C`.

In [1]:
from pathlib import Path
import importlib.metadata
import pandas as pd
import numpy as np

try:
    streamlit_version = importlib.metadata.version("streamlit")
except importlib.metadata.PackageNotFoundError:
    streamlit_version = "not installed"

print("Streamlit version:", streamlit_version)

Streamlit version: 1.58.0


## 2. Smallest app

Save this as `hello.py`:

```python
import streamlit as st

st.set_page_config(page_title="Hello")
st.title("Hello Streamlit")
st.write("My first data app!")
```

Run `streamlit run hello.py`.

### Step by step

1. Import Streamlit with the common nickname `st`.
2. Configure the browser tab before other page elements.
3. Add a title.
4. Add normal content.
5. Streamlit serves the generated interface and watches widget events.

## 3. The rerun model — the most important idea

Each browser interaction generally asks Streamlit to run the script again from top to bottom.

```text
visitor changes widget
        ↓
Streamlit updates widget value
        ↓
script reruns top → bottom
        ↓
page is redrawn
```

Consequences:

- ordinary local variables start again;
- widget values are restored by Streamlit;
- use session state for per-session memory;
- cache expensive deterministic work;
- put irreversible side effects behind deliberate submit actions and make them safe to repeat.

## 4. Text elements

| Command | Use |
|---|---|
| `st.title()` | page title |
| `st.header()` | major section |
| `st.subheader()` | smaller section |
| `st.write()` | general-purpose output |
| `st.markdown()` | formatted notes |
| `st.caption()` | small context/caveat |
| `st.code()` | highlighted source |
| `st.latex()` | math equation |

```python
st.title("Sales Explorer")
st.markdown("Choose a **region** to begin.")
st.caption("Revenue shown in USD.")
```

Avoid unsafe HTML for untrusted content. Let Streamlit/Markdown handle output normally.

## 5. Widgets and inputs

A widget is a question your page asks the visitor.

```python
name = st.text_input("Name", key="name")
age = st.number_input("Age", min_value=0, max_value=120)
language = st.selectbox("Language", ["Python", "Java", "C++"])
topics = st.multiselect("Topics", ["Charts", "State", "Caching"])
ready = st.checkbox("Ready")

if st.button("Greet"):
    st.success(f"Hello {name}")
```

Keys must be unique and stable when labels repeat. A button is `True` for the rerun triggered by its click; it is not lasting state.

## 6. Forms batch inputs

Without a form, changing most widgets reruns immediately. A form collects values until submit.

```python
with st.form("profile"):
    name = st.text_input("Name")
    age = st.slider("Age", 0, 120, 18)
    submitted = st.form_submit_button("Save")

if submitted:
    if not name.strip():
        st.error("Name is required")
    else:
        st.success("Saved")
```

Browser widget limits improve usability, but validate again in Python. A client can send unexpected input.

## 7. Layout: sidebar, columns, tabs, containers

```python
with st.sidebar:
    region = st.selectbox("Region", regions)

left, right = st.columns(2)
left.metric("Sales", "$12,000", "+8%")
right.metric("Orders", "320")

overview, rows = st.tabs(["Overview", "Rows"])
with overview:
    st.line_chart(daily_sales)
with rows:
    st.dataframe(filtered_data)

with st.expander("Method"):
    st.write("How the metric was calculated")
```

Tabs organize display, but code inside unselected tabs may still execute. Cache or guard expensive operations instead of assuming hidden means skipped.

## 8. Session state

Session state is one browser session's backpack.

```python
if "count" not in st.session_state:
    st.session_state.count = 0

if st.button("Add"):
    st.session_state.count += 1

st.metric("Count", st.session_state.count)
```

Use it for wizard steps, selected records, counters, and temporary user work. It is not a durable database, may disappear with the session/server, and should not contain secrets or huge objects.

In [2]:
# Pure-Python version of the initialization idea used by session state.
state = {}
state.setdefault("count", 0)
state["count"] += 1
print(state)

{'count': 1}


## 9. Caching

| Decorator | Good for | Important behavior |
|---|---|---|
| `@st.cache_data` | DataFrames, API results, calculations | serializes/copies results for callers |
| `@st.cache_resource` | model, database client, shared resource | shared object; mutable/thread-safety concerns |

```python
@st.cache_data(ttl=300, max_entries=20)
def load_csv(path):
    return pd.read_csv(path)

@st.cache_resource
def load_model():
    return train_model()
```

Cache keys depend on function code and hashable arguments. Use TTL/entry limits when data changes or arguments can grow. Never cache per-user private results in a globally shared way without carefully including identity and access rules.

## 10. Pandas and the repository CSV

`sampledata.csv` contains the original John, Jane, Jake, and Jill example. Resolve paths from the script file in apps so launching from another directory still works.

In [3]:
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "sampledata.csv").exists():
    candidate = NOTEBOOK_DIR / "Complete-Python-Bootcamp-main" / "14-Streamlit"
    if candidate.exists():
        NOTEBOOK_DIR = candidate

sample_path = NOTEBOOK_DIR / "sampledata.csv"
people = pd.read_csv(sample_path, index_col=0)
people

,Name,Age,City
0,John,28,New York
1,Jane,24,Los Angeles
2,Jake,35,Chicago
3,Jill,40,Houston


### Tables

```python
st.dataframe(people, width="stretch")  # interactive
st.table(people.head())                         # static
edited = st.data_editor(people)                 # editable result
```

Rendering huge tables is slow and not useful. Show a filtered preview, paginate, aggregate, or download the complete result. Validate edits before saving them.

In [4]:
summary = people.agg(
    people=("Name", "count"),
    average_age=("Age", "mean"),
    cities=("City", "nunique"),
)
summary

,Name,Age,City
people,4.0,NaN,NaN
average_age,NaN,31.75,NaN
cities,NaN,NaN,4.0


## 11. CSV upload and download

```python
uploaded = st.file_uploader("CSV", type="csv")
if uploaded is not None:
    try:
        frame = pd.read_csv(uploaded)
    except (UnicodeDecodeError, pd.errors.ParserError) as error:
        st.error(f"Cannot read file: {error}")
    else:
        st.dataframe(frame.head(500))
        st.download_button("Download", frame.to_csv(index=False), "result.csv")
```

The extension filter is not a security guarantee. Limit sizes, parse defensively, avoid using uploaded filenames as trusted paths, and never execute uploaded content.

## 12. Charts

Quick charts:

```python
st.line_chart(frame, x="date", y="sales")
st.bar_chart(summary, x="region", y="revenue")
st.scatter_chart(frame, x="age", y="score", color="group")
```

For more control, build an Altair chart and send it to `st.altair_chart`. Use tidy data, clear labels/units, honest axes, useful tooltips, and an explanation of filters/assumptions.

In [5]:
rng = np.random.default_rng(42)
chart_frame = pd.DataFrame(
    {
        "day": pd.date_range("2025-01-01", periods=7),
        "sales": rng.integers(80, 160, 7),
    }
)
chart_frame

,day,sales
0,2025-01-01,87
1,2025-01-02,141
2,2025-01-03,132
3,2025-01-04,115
4,2025-01-05,114
5,2025-01-06,148
6,2025-01-07,86


## 13. Images, metrics, and status elements

```python
st.image("assets/streamlit_learning.svg", caption="Local asset")
st.metric("Revenue", "$12K", delta="8%")
st.success("Saved")
st.info("Read this")
st.warning("Check the input")
st.error("Could not continue")

with st.spinner("Working..."):
    result = slow_job()

bar = st.progress(0)
bar.progress(0.5, text="Halfway")
placeholder = st.empty()
placeholder.write("This region can be replaced")
```

Progress must reflect real work when possible. Do not animate fake certainty for an unpredictable task.

## 14. ML model integration

The improved `classification.py` and `projects/ml_prediction.py` show this flow:

1. load data with `cache_data`;
2. train/load model with `cache_resource`;
3. collect named features in the exact expected order;
4. validate feature ranges/types;
5. predict;
6. show class probabilities and limitations.

Production ML also needs preprocessing parity, versioning, evaluation, drift monitoring, privacy/fairness review, safe failure behavior, and human oversight where decisions matter.

In [6]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

iris = load_iris(as_frame=True)
model = LogisticRegression(max_iter=500, random_state=42)
model.fit(iris.data, iris.target)
example = iris.data.iloc[[0]]
prediction = int(model.predict(example)[0])
probability = float(model.predict_proba(example).max())
print("Class:", iris.target_names[prediction])
print("Largest model probability:", round(probability, 3))

Class: setosa
Largest model probability: 0.982


## 15. Multipage apps

This folder contains:

```text
multipage_app/
├── Home.py
└── pages/
    ├── 1_Data.py
    ├── 2_Charts.py
    └── 3_About.py
```

Run:

```text
streamlit run multipage_app/Home.py
```

The pages directory is the simplest approach. Programmatic navigation offers more control in larger apps. Put shared pure functions/data loaders in importable modules rather than importing page scripts that render immediately.

## 16. Theme and configuration

`.streamlit/config.toml` stores harmless project defaults such as theme colors and upload limits.

```toml
[theme]
primaryColor = "#2563EB"
backgroundColor = "#FFFFFF"
```

Configuration is not a secret vault. Keep API keys/passwords out of Git and use environment variables or the hosting platform's secret management. If using `st.secrets`, document required key names without committing real values.

## 17. Deployment

Deployment checklist:

- pick the correct entry `.py` file;
- install pinned/tested dependencies;
- make file paths independent of launch directory;
- store secrets outside source;
- cache expensive work with sensible invalidation;
- use persistent storage for durable/shared data;
- add authentication/authorization for private data;
- handle empty/error/slow external services;
- monitor failures, latency, memory, and data/model quality;
- test on the same Python/dependency versions used by the host.

Hosting steps change over time. Verify the current official instructions of Streamlit Community Cloud or your chosen container/Python platform when publishing.

## 18. Common mistakes

| Mistake | Better habit |
|---|---|
| expecting local variables to survive | session state or persistent store |
| retraining/reloading on every rerun | correct cache decorator |
| caching without TTL/bounds | define invalidation and growth limits |
| writing files on every widget change | use deliberate form/button action |
| trusting uploaded extension/name | parse, limit, validate, isolate |
| global list for multi-user records | database/service |
| duplicate widget keys | unique stable keys |
| huge DataFrame rendered at once | aggregate/filter/preview/download |
| secrets in source | environment/platform secrets |
| misleading metrics/charts | define units, filters, baseline, caveats |
| model confidence treated as truth | show evaluation and limitations |

## 19. Mini-project map

| Project | File | Main concepts |
|---|---|---|
| Sales dashboard | `projects/dashboard.py` | filters, KPIs, charts |
| CSV explorer | `projects/csv_explorer.py` | upload, quality summary, export |
| Feedback form | `projects/form_app.py` | forms, validation, session state |
| Interactive charts | `projects/interactive_charts.py` | Altair, tooltips, filters |
| ML prediction | `projects/ml_prediction.py` | caching, model, probabilities |
| Data analysis dashboard | `projects/data_analysis_dashboard.py` | repository CSV, Pandas, KPIs |

Build them in this order, then complete `PRACTICE.md`.

# Quick Revision Cheat Sheet

```python
import streamlit as st

st.set_page_config(page_title="App", layout="wide")
st.title("App")

with st.sidebar:
    choice = st.selectbox("Choice", options)

with st.form("inputs"):
    value = st.number_input("Value")
    submitted = st.form_submit_button("Run")

if "count" not in st.session_state:
    st.session_state.count = 0

@st.cache_data(ttl=300)
def load_data(path): ...

@st.cache_resource
def load_model(): ...

st.metric("Rows", len(data))
st.dataframe(data)
st.line_chart(data)
```

## Remember

- Streamlit reruns from top to bottom after widget changes.
- Forms batch values; session state remembers one session; caches reuse computed data/resources.
- Validate every upload and model input.
- Use Pandas for preparation, then display bounded tables and honest charts.
- Use pages/navigation for larger apps and configuration for themes/defaults.
- Durable/shared information belongs in a database, not a global variable.
- Deploy with reproducible dependencies, protected secrets, error handling, and monitoring.